In [17]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

# -------------------------
# CONFIG (edit as needed)
# -------------------------
C_FILE  = Path(r"C:\Drought\Regridding and data clipping\VODC")  / "VODCA_Cband_monthly_India_0p05deg_2003-2018.nc"
KU_FILE = Path(r"C:\Drought\Regridding and data clipping\VODKu") / "VODCA_Kuband_monthly_India_0p05deg_2000-2016.nc"
X_FILE  = Path(r"C:\Drought\Regridding and data clipping\VODX")  / "VODCA_Xband_monthly_India_0p05deg_2000-2018.nc"

OUT_NC = Path(r"C:\Drought\Regridding and data clipping\VOD") / "VOD_Z_Fused_monthly_India_0p05deg_2000present_v2.nc"
OUT_CSV = Path(r"C:\Drought\Regridding and data clipping\VOD") / "VOD_Z_Fused_monthly_India_mean_timeseries.csv"

BASE_START = "2003-01"   # triple-overlap start
BASE_END   = "2016-12"   # triple-overlap end
CLIP_Z     = (-3.0, 3.0)

# fusion weights (renormalized where bands are missing)
W_X  = 0.60
W_C  = 0.30
W_KU = 0.10

CHUNKS = {"time": 36, "lat": 300, "lon": 300}

# -------------------------
# Helpers
# -------------------------
def to_month_end(da: xr.DataArray) -> xr.DataArray:
    t = pd.to_datetime(da["time"].values)
    mend = pd.PeriodIndex(t, freq="M").to_timestamp("M")
    da2 = da.assign_coords(time=mend)
    # drop duplicates, sort
    _, idx = np.unique(da2["time"].values, return_index=True)
    return da2.isel(time=np.sort(idx)).sortby("time")

def monthwise_mu_sd(da: xr.DataArray, base_start: str, base_end: str):
    base = da.sel(time=slice(base_start, base_end))
    mus, sds = [], []
    for m in range(1, 13):
        bm = base.where(base["time.month"] == m, drop=True)
        mus.append(bm.mean("time", skipna=True))
        sds.append(bm.std("time", ddof=1, skipna=True))
    mu = xr.concat(mus, dim="month").assign_coords(month=np.arange(1, 13))
    sd = xr.concat(sds, dim="month").assign_coords(month=np.arange(1, 13))
    return mu, sd

def apply_month_index(da: xr.DataArray, aux_by_month: xr.DataArray):
    idx = xr.DataArray(da["time.month"].values, coords={"time": da["time"]}, dims=("time",))
    return aux_by_month.sel(month=idx)

def load_vod(path: Path, varname_hint="vod") -> xr.DataArray:
    ds = xr.open_dataset(path)
    varname = varname_hint if varname_hint in ds else list(ds.data_vars)[-1]
    da = ds[varname]
    da = to_month_end(da)
    return da

def ensure_same_grid(template: xr.DataArray, other: xr.DataArray) -> xr.DataArray:
    """Reindex 'other' to 'template' lat/lon if not equal."""
    for ax in ("lat", "lon"):
        if not np.array_equal(template[ax].values, other[ax].values):
            other = other.reindex({ax: template[ax].values}, method=None)
    return other

def zscore_by_month(da: xr.DataArray, bstart: str, bend: str, clip=CLIP_Z) -> xr.DataArray:
    mu_m, sd_m = monthwise_mu_sd(da, bstart, bend)
    mu_t = apply_month_index(da, mu_m)
    sd_t = apply_month_index(da, sd_m)
    z = ((da - mu_t) / sd_t).clip(*clip)
    return z

def weighted_fuse(vars_z: dict, weights: dict) -> xr.DataArray:
    """
    Fuse available z-layers with renormalized weights.
    vars_z: dict of {'X': zX, 'C': zC, 'Ku': zKu} (any subset allowed)
    weights: dict of base weights
    """
    # stack available
    zs = []
    ws = []
    for k, z in vars_z.items():
        if z is None:
            continue
        zs.append(z)
        ws.append(weights[k])
    if len(zs) == 0:
        return None
    # renormalize weights where some bands are missing at a given (t,lat,lon)
    # approach: compute per-pixel sum of weight masks
    # stack into one dataset for clean broadcasting
    ds_tmp = xr.Dataset({f"z_{i}": zz for i, zz in enumerate(zs)})
    weight_vals = xr.concat([xr.ones_like(zs[0]) * w for w in ws], dim="w")
    avail_mask = xr.concat([xr.where(np.isfinite(zz), 1.0, 0.0) for zz in zs], dim="w")
    wsum = (weight_vals * avail_mask).sum("w")
    # prevent division by zero
    wnorm = xr.where(wsum > 0, (weight_vals / wsum) * avail_mask, 0.0)
    zstack = xr.concat(zs, dim="w")
    fused = (wnorm * zstack).sum("w")
    fused = fused.rename("VOD_fused_z").assign_attrs(
        long_name="Fused VOD Z (weights renormalized on availability)",
        units="1"
    )
    return fused

def jackknife_uncert(vars_z: dict) -> xr.DataArray:
    """Uncertainty = std dev across available band Z's (simple spread)."""
    zs = [z for z in vars_z.values() if z is not None]
    if len(zs) == 0:
        return None
    zstack = xr.concat(zs, dim="w")
    return zstack.std("w", skipna=True).rename("VOD_uncert").assign_attrs(
        long_name="Inter-band spread (std) of available VOD Z-scores",
        units="1"
    )

# -------------------------
# Load & harmonize
# -------------------------
print("Loading VOD bands …")
vod_c  = load_vod(C_FILE,  varname_hint="vod")
vod_ku = load_vod(KU_FILE, varname_hint="vod")
vod_x  = load_vod(X_FILE,  varname_hint="vod")

# pick a template grid (X-band)
tmpl = vod_x
vod_c  = ensure_same_grid(tmpl, vod_c)
vod_ku = ensure_same_grid(tmpl, vod_ku)

# unify time axis = union of all times
all_time = np.union1d(np.union1d(vod_c.time.values, vod_ku.time.values), vod_x.time.values)
vod_c  = vod_c.reindex(time=all_time)
vod_ku = vod_ku.reindex(time=all_time)
vod_x  = vod_x.reindex(time=all_time)

# -------------------------
# Per-band Z (calendar-month, common baseline 2003–2016)
# -------------------------
print(f"Computing Z-scores vs baseline {BASE_START}..{BASE_END} …")
C_z  = zscore_by_month(vod_c,  BASE_START, BASE_END).rename("VOD_C_z").assign_attrs(band="C")
Ku_z = zscore_by_month(vod_ku, BASE_START, BASE_END).rename("VOD_Ku_z").assign_attrs(band="Ku")
X_z  = zscore_by_month(vod_x,  BASE_START, BASE_END).rename("VOD_X_z").assign_attrs(band="X")

# -------------------------
# Fusion & uncertainty
# -------------------------
print("Fusing bands with weights (X=0.6, C=0.3, Ku=0.1) …")
weights = {"X": W_X, "C": W_C, "Ku": W_KU}
vars_z  = {"X": X_z, "C": C_z, "Ku": Ku_z}

VOD_fused_z = weighted_fuse(vars_z, weights)
VOD_uncert  = jackknife_uncert(vars_z)

# -------------------------
# Save gridded NetCDF
# -------------------------
print(f"Writing NetCDF → {OUT_NC} …")
attrs_ds = dict(
    title="VOD Z-scores (C, Ku, X) and fused VOD_z for India (0.05°)",
    summary=f"Per-band monthly Z-scores vs {BASE_START}–{BASE_END} (calendar-month). "
            f"Fused VOD_z uses weights X={W_X}, C={W_C}, Ku={W_KU} with availability-based renormalization. "
            f"Uncertainty = std across available bands.",
    conventions="CF-1.8",
    crs="EPSG:4326",
    spatial_resolution="0.05 degree",
    temporal_aggregation="monthly",
    baseline=f"{BASE_START}..{BASE_END}",
    history="Generated by make_vod_z_fused.py",
)
ds_out = xr.Dataset(
    data_vars=dict(
        VOD_C_z=C_z,
        VOD_Ku_z=Ku_z,
        VOD_X_z=X_z,
        VOD_fused_z=VOD_fused_z,
        VOD_uncert=VOD_uncert,
    ),
    attrs=attrs_ds,
)
comp = dict(zlib=True, complevel=4)
encoding = {v: comp for v in ds_out.data_vars}
encoding.update({"lat": {"dtype": "float32"}, "lon": {"dtype": "float32"}})
ds_out.to_netcdf(OUT_NC, encoding=encoding)

# -------------------------
# Save India-mean time series CSV
# -------------------------
print(f"Writing CSV (India-mean) → {OUT_CSV} …")
# simple area-weighted mean: weight by cos(lat)
lat = ds_out["lat"]
w = np.cos(np.deg2rad(lat))
w = w / w.mean()  # normalize to ~1 on average; only relative scaling matters

def area_mean(da):
    # weights across lat only; xarray will broadcast along lon
    return (da.weighted(w).mean(("lat", "lon"))).to_series()

df = pd.DataFrame({
    "time": pd.to_datetime(ds_out.time.values)
})
for name in ["VOD_C_z", "VOD_Ku_z", "VOD_X_z", "VOD_fused_z", "VOD_uncert"]:
    if name in ds_out:
        s = area_mean(ds_out[name])
        df[name] = s.reindex(df["time"]).values

df.to_csv(OUT_CSV, index=False)

print("Done.")


Loading VOD bands …
Computing Z-scores vs baseline 2003-01..2016-12 …
Fusing bands with weights (X=0.6, C=0.3, Ku=0.1) …
Writing NetCDF → C:\Drought\Regridding and data clipping\VOD\VOD_Z_Fused_monthly_India_0p05deg_2000present_v2.nc …
Writing CSV (India-mean) → C:\Drought\Regridding and data clipping\VOD\VOD_Z_Fused_monthly_India_mean_timeseries.csv …
Done.
